# Imports Necessários

In [ ]:
from numpy import full, ndenumerate
from os.path import isfile, isdir
from cv2.typing import MatLike
from itertools import product
from os import mkdir, system
from cv2 import (
  imread, imwrite, cvtColor,
  adaptiveThreshold, Canny,
  GaussianBlur, resize,
  ADAPTIVE_THRESH_GAUSSIAN_C,
  COLOR_BGR2GRAY,
  COLOR_GRAY2BGR,
  THRESH_BINARY
)

In [ ]:
BLACK, WHITE, RED = 0, 255, (0, 0, 255)
tuple2int = tuple[int, int]
listTuple = list[tuple2int]
listsTuple = list[listTuple]
shape = (1820, 4760)

In [ ]:
def bgr2gray(entrie: MatLike):
  return cvtColor(entrie, COLOR_BGR2GRAY)

def gray2bgr(entrie: MatLike):
  return cvtColor(entrie, COLOR_GRAY2BGR)

Função para abrir imagens:

In [ ]:
def imopen(entrie: str):
  if not isfile(entrie):
    exit(f"\"{entrie}\" inexistente!")
  name, exten = entrie.split(".")
  temp = imread(entrie)
  if exten != "png":
    imwrite(f"{name}/{name}.png", temp)
  return name, bgr2gray(temp)

Função para cortar as imagens:

![](3imagem\\3imagem.png)
Imagem a ser cortada

In [ ]:
def cutImage(entrie: MatLike, fator = 10):
  ALTR, LARG = entrie.shape
  tamanho = (LARG//fator, ALTR//fator)
  res = resize(entrie, tamanho)
  res = gaussThresh(res, 9, 9)
  res = cannyFilter(res)
  res = estimateBack(res, 380, 10)[1]
  res = xFilter2(res, 10)
  res = organize(extract(res), 5, 350)
  res.sort(key = lambda x: x[0][0])
  altr, larg = res[0][len(res[0])//2]
  altr, larg = altr*fator, larg*fator
  x1 = altr - round((124/267)*ALTR)
  x2 = altr + round((197/534)*ALTR)
  y1 = larg - round((589/1250)*LARG)
  y2 = larg + round((561/1250)*LARG)
  cutted = entrie[x1:x2, y1:y2]
  cutted = resize(cutted, shape[::-1])
  print("cutImage terminado!")
  return cutted

Resultado do corte:

![](3imagem\\cutted.png)

A imagem acima vai ser processada nas próximas 2 funções, que são responsáveis por ajudar na demarcação das linhas

In [ ]:
def gaussThresh(entrie: MatLike, size = 19, C = 10):
  output = adaptiveThreshold(entrie, 255,
    ADAPTIVE_THRESH_GAUSSIAN_C, THRESH_BINARY, size, C)
  print("gaussThresh terminado!")
  return output

def gaussBlur(entrie: MatLike):
  output = GaussianBlur(entrie, (11, 11), 0)
  print("gaussThresh terminado!")
  return output

Resultado das funções anteriores:

![](3imagem\\gauss.png)

Agora, temos que processá-la pelo filtro canny, o que faz a maior parte do processamento:

In [ ]:
def cannyFilter(entrie: MatLike):
  output = Canny(entrie, 200, 255, L2gradient = True)
  print("cannyFilter terminado!")
  return output

Resultado da função anterior:

![](3imagem\\canny.png)

Agora, temos que diferenciar as linhas de referência das linhas de variação:

In [ ]:
def estimateBack(entrie: MatLike, quant = 1200, proc = 15):
  deleted = entrie.copy()
  estimated = full(entrie.shape, BLACK, "u1")
  lista = [[(i1, i2) for i2, elem in enumerate(arr) if elem]
            for i1, arr in enumerate(entrie == WHITE)]
  index: list[int] = []
  atual, lentmp = 0, len(lista)
  while True:
    meio = lentmp
    for i in range(atual, lentmp):
      if len(lista[i]) > quant:
        meio = i; break
    if meio == lentmp: break
    raio = meio - proc
    for i in range(raio, meio):
      e1 = len(lista[i])
      e2 = len(lista[i+1])
      if (e2 - e1) > (quant//12):
        pos1 = i; break
    raio = meio + proc
    for i in range(raio, meio, -1):
      e1 = len(lista[i])
      e2 = len(lista[i-1])
      if (e2 - e1) > (quant//12):
        pos2 = i; break
    deleted[pos1:pos2, :] = BLACK
    pos2 += 1; atual = raio
    index.extend(range(pos1, pos2))
  dim1, dim2 = zip(*(e for x in index for e in lista[x]))
  estimated[dim1, dim2] = WHITE
  print("estimateBack terminado!")
  return deleted, estimated

Resultado dessa função:

![](3imagem\\deleted.png)
![](3imagem\\estimat.png)

Agora, cada uma dessas imagens deve passar pela função de esqueletização:

In [ ]:
def xFilter2(entrie: MatLike, dist = 20):
  output = entrie.copy()
  linhas, colunas = output.shape
  lentrie = linhas - dist
  for y in range(colunas):
    pos1, flag = 0, True
    while True:
      flag = False
      for a in range(pos1, lentrie):
        if output[a, y] == WHITE:
          pos1, flag = a, True
          break
      if not flag: break
      pos2 = pos1 + dist
      for b in range(pos2, pos1, -1):
        if output[b, y] == WHITE:
          pos2 = b + 1
          break
      output[pos1:pos2, y] = BLACK
      temp, pos1 = (a + b)//2, pos2
      output[temp, y] = WHITE
  print("xFilter2 terminado!")
  return output

Resultados de ambos processamentos:

![](3imagem\\filter1.png)
![](3imagem\\filter2.png)

Agora, temos que extrair e organizar os pixels de ambas as imagens, deixando somente o necessário.

In [ ]:
def extract(entrie: MatLike):
  elems = ndenumerate(entrie == WHITE)
  output = [idx for idx, elem in elems if elem]
  output.sort(); print("extract terminado!")
  return output

def juntarImgs(listas: listsTuple):
  output = full(shape, BLACK, "u1")
  dim1, dim2 = zip(*(elem for sub in listas for elem in sub))
  output[dim1, dim2] = WHITE; print("juntarImgs terminado!")
  return output

def organize(tuplas: listTuple, radius = 10, minlen = 1200):
  DIST, TAM = 10, 60
  listas: listsTuple = []
  pos1 = 0; lentup = len(tuplas)
  while pos1 != lentup:
    temp = [tuplas[pos1]]
    pos1 += 1; pos2 = lentup
    for i in range(pos1, lentup):
      x1 = tuplas[i][0]
      x2 = temp[-1][0]
      if (x1 - x2) > radius:
        pos2 = i; break
      temp.append(tuplas[i])
    listas.append(temp)
    pos1 = pos2
  listas.sort(key = lambda x: len(x), reverse = True)
  for elem in listas: elem.sort(key = lambda x: x[1])
  list1: listsTuple = []
  for sublista in listas:
    temp = [sublista[0]]
    for tupla in sublista[1:]:
      res1 = tupla[1] - temp[-1][1]
      if res1 > DIST and len(temp) < TAM:
        temp.clear()
      temp.append(tupla)
    list1.append(temp)
  list2: listsTuple = []
  for sublista in list1:
    temp = [sublista[0]]
    for tupla in sublista[1:]:
      res1 = tupla[0] - temp[-1][0]
      res2 = tupla[1] != temp[-1][1]
      if abs(res1) <= DIST and res2:
        temp.append(tupla)
    list2.append(temp)
  listas = [elem for elem in list2 if len(elem) > minlen]
  listas.sort(); print("organize terminado!")
  return listas

Resultado desses processamentos:

![](3imagem\processImg\\base0.png)
Linha de base 0

![](3imagem\processImg\\base1.png)
Linha de base 1

![](3imagem\processImg\\curve0.png)
Curva de medição 0

![](3imagem\processImg\\curve1.png)
Curva de medição 1

![](3imagem\processImg\\curve2.png)
Curva de medição 2

Lembre-se que cada imagem tem seu respectivo arquivo de coordenadas, sendo possível calcular distâncias dos pixels!

Abaixo, temos uma função que apenas salva os resultados anteriores.

In [ ]:
def saveLists(listas: listsTuple, dtype: str, dname: str):
  preta = full(shape, BLACK, "u1")
  pasta = f"{dname}/processImg"
  if not isdir(pasta): mkdir(pasta)
  for i, sublista in enumerate(listas):
    imagem = preta.copy()
    with open(f"{pasta}/{dtype}{i}.txt", "w") as saida:
      saida.write(f"Tam.: {shape}\n")
      for tupla in sublista:
        saida.write(f"{tupla}\n")
        imagem[tupla] = WHITE
    imwrite(f"{pasta}/{dtype}{i}.png", imagem)
  print("saveLists terminado!")

Aqui, a função apenas sobrepõe os resultados na imagem de extração de esqueletos.

![](3imagem\\sobre1.png)

In [ ]:
def sobrepor(canny: MatLike, listas: listsTuple):
  output = gray2bgr(canny)
  dim1, dim2 = zip(*(elem for sub in listas for elem in sub))
  output[dim1, dim2] = RED; print("sobrepor terminado!")
  return output

Abaixo, temos a função em que estamos travados no momento.\
Ela é responsável por converter as diferenças entre as coordenadas dos pixels em milímetros, por enquanto.

In [ ]:
alt = shape[0]
h_pixel, d_pixel, z_pixel = (460/alt), (46/alt), (460/alt)
variations = (h_pixel, d_pixel, z_pixel)

def calcDiff(dname: str, curves: listsTuple, basels: listsTuple):
  pasta = f"{dname}/calculoDiff"
  if not isdir(pasta): mkdir(pasta)
  ncurves = enumerate(zip(curves, variations))
  nbasels = enumerate(basels)
  for (a, (sub1, var)), (b, sub2) in product(ncurves, nbasels):
    with open(f"{pasta}/curve{a}base{b}.txt", "w") as saida:
      pos2, len2 = 0, len(sub2)
      for el1 in sub1:
        e1, e2 = el1
        for i in range(pos2, len2):
          e3, e4 = el2 = sub2[i]
          if e2 == e4:
            pos2, value = (i + 1), var*(e1 - e3)
            saida.write(f"{el1}{el2} - {value}\n")
            break
  print("calcDiff terminado!")

Como exemplo, temos os seguintes arquivos:

![](3imagem\\exemplo.png)

Esses arquivos comparam a linha de base 0 com as curvas 0 e 1, respectivamente.
Note que esses arquivos são suficientemente longos, com cerca de 4000 linhas cada.

In [ ]:
system("cls||clear")
name, output = imopen(input("Entrie archive: "))
if not isdir(name): mkdir(name)

output = cutImage(output)
imwrite(f"{name}/cutted.png", output)
output = gaussThresh(output)
output = gaussBlur(output)
imwrite(f"{name}/gauss.png", output)
canny = cannyFilter(output)
imwrite(f"{name}/canny.png", canny)

deleted, estimat = estimateBack(canny)
imwrite(f"{name}/estimat.png", estimat)
imwrite(f"{name}/deleted.png", deleted)

deleted, estimat = xFilter2(deleted), xFilter2(estimat)
imwrite(f"{name}/filter1.png", deleted)
imwrite(f"{name}/filter2.png", estimat)

lista1, lista2 = extract(deleted), extract(estimat)
output = juntarImgs([lista1, lista2])
imwrite(f"{name}/juntos.png", output)

lista1, lista2 = organize(lista1), organize(lista2)
saveLists(lista1, "curve", name)
saveLists(lista2, "base", name)
calcDiff(name, lista1, lista2)

saida1 = sobrepor(output, lista1)
saida2 = sobrepor(canny, lista1)
imwrite(f"{name}/sobre1.png", saida1)
imwrite(f"{name}/sobre2.png", saida2)

print("Processo Terminado!")